# 실습 0 — 환경 구성

실습 1~3 을 수행할 수 있는 환경을 구성하고, 그 환경이 정상인지 스모크 런 두 번으로 검증한다.

## 학습 목표

1. nuPlan devkit·모델·MPC 솔버를 conda 환경 `e2e_refinement` 에 설치한다
2. 데이터(로그 DB·지도·체크포인트)의 배치를 확인한다
3. 시뮬레이션과 채점이 정상 동작하는지 점수로 검증한다

## 실습 결과물

| 확인 대상 | 단계 | 정상 기준 |
|---|---|---|
| 파이썬 패키지 | 6 | 검증 대상 import 전부 성공 |
| MPC 솔버 `.so` | 5 | 빌드 성공 *(실습 3 에서 사용)* |
| 데이터 배치 | 7·8 | 시나리오 5 개 생성 |
| 시뮬레이션·채점 | 9 | `log_future_planner` **1.000** / `simple_planner` **0.000** |
| 모델·MPC 경로 | 10 | 영상과 스텝별 점수 생성 |

9 단계의 두 값이 본 노트북의 합격 기준이다. `log_future_planner` 는 로그의 실제 궤적을 그대로
출력하므로 Open-loop 점수가 정의상 1.000 이어야 하며, 다른 값이 나오면 채점 또는 집계에 문제가
있는 것이다.

## 전체 실습 구성에서의 위치

```
 실습 0            실습 1               실습 2                실습 3
 환경 구성      →  nuPlan 프레임워크  →  ML planner 평가    →  MPC refinement
 데이터 배치        구성 요소와 교체      Open-loop·Closed-loop  궤적 후처리와 비교
```

## 진행 순서

| 단계 | 내용 | 대략 소요 |
|---|---|---|
| 1 | 시스템 점검 (OS / 컴파일러 / GPU / 디스크) | 즉시 |
| 2 | 파이썬 패키지 설치 | ~10 분 |
| 3 | natten 설치 (PLUTO 의 Attention 연산) | ~1 분 |
| 4 | acados 설치 | ~5 분 |
| 5 | MPC 솔버 빌드 | ~1 분 |
| 6 | 설치 검증 | 즉시 |
| 7 | 데이터 배치 확인 | 즉시 |
| 8 | 시나리오 빌드 확인 | ~1 분 |
| 9 | Open-loop 스모크 런 (devkit 기본 planner) | ~1 분 |
| 10 | 모델 스모크 런 (실습 2 이후) | ~3 분 |

- conda 와 환경 `e2e_refinement` 는 **README 의 「준비」** 에서 생성한 것을 전제한다.
- **본 노트북의 커널은 실습 환경이 아니어도 된다.** 모든 설치를 subprocess 로 실행한다.
  실습 1 부터는 커널 내에서 devkit 을 직접 import 하므로 `E2E Refinement` 커널이 필요하다.
- **실습 1 만 수행하려면 4·5·10 단계를 생략해도 된다.** 9 단계까지 통과하면 시작할 수 있다.
- 단계 2~5 는 `bash script/setup_env.sh` 한 줄과 거의 동일하다(해당 스크립트는 환경 생성까지 포함한다).

---

## 검증 파이프라인과 설치 단계의 대응

실습이 한 스텝에서 하는 일은 여섯 단계다. 각 설치 단계가 이 중 어디를 담당하는지 먼저 보면
"이 패키지가 왜 필요한가" 가 분명해진다.

```
 [A] 시나리오 로드      DB → 지도 → nuPlan scenario 객체
       ↓
 [B] ML 추론            PLUTO/Diffusion → ego-local 궤적 (80,3)
       ↓
 [C] MPC refinement     궤적을 reference 로 acados 를 풀어 개선 궤적
       ↓
 [D] 채점               두 궤적을 같은 조건에서 nuPlan 공식 metric 으로
       ↓
 [E] 렌더               지도·agent·궤적·점수표를 프레임으로
       ↓
 [F] 집계               스텝별 CSV + 런 단위 공식 점수(parquet)
```

| 파이프라인 | 담당 패키지 | 설치 단계 |
|---|---|---|
| [A] 시나리오 로드 | `sqlalchemy`(DB ORM) · `geopandas`/`pyogrio`/`rasterio`/`shapely`(지도) · `hydra-core`(설정 조합) · `ray`(병렬 워커) | 2 |
| [B] ML 추론 | `torch+cu118` · `timm` · **`natten`** · `numba`/`cv2`(feature builder) · `pytorch-lightning`(ckpt 로드) · `tensorboard`(devkit callback) | 2, 3 |
| [C] MPC refinement | **`acados`(C 라이브러리)** · `acados_template` · `casadi` · `t_renderer` | 4, 5 |
| [D] 채점 | devkit metric 클래스 + `numpy`/`scipy`(forward simulation) | 2 |
| [E] 렌더 | `matplotlib` · `Pillow` · `imageio` + **`imageio-ffmpeg`**(mp4 인코딩) | 2 |
| [F] 집계 | `pandas` · `pyarrow`(parquet) | 2 |


## 0. 준비 — 환경 확인 및 헬퍼 로드

conda 설치와 실습 환경 `e2e_refinement` 생성, 노트북 커널 등록은 **README 의 「준비」** 에서
이미 마쳤다. 이 노트북은 그 환경 위에 devkit·모델·솔버를 올리는 일부터 시작한다.

아래 셀은 저장소 루트와 헬퍼를 찾고, 그 환경이 실제로 준비되었는지 확인한다. 환경을 찾지
못하면 README 의 1·2 절을 먼저 수행한다.


In [ ]:
import sys, pathlib

# 저장소 루트를 기준으로 잡는다 (노트북을 어디서 열든 동작하도록).
NB_DIR = pathlib.Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "run_simulation.py").exists() else NB_DIR.parent
assert (REPO_ROOT / "run_simulation.py").exists(), f"저장소 루트를 찾지 못했습니다: {NB_DIR}"
# 헬퍼 위치를 이름으로 찾는다 (practice/ 아래 폴더 구조가 바뀌어도 동작하도록).
helper = next(REPO_ROOT.glob("practice/**/_setup_helper.py"))
sys.path.insert(0, str(helper.parent))

# 헬퍼가 수정되면 커널이 캐시한 옛 모듈을 계속 쓴다. 이 셀을 다시 실행하면
# 최신 내용을 읽도록 reload 를 거친다.
import importlib
import _setup_helper
importlib.reload(_setup_helper)
from _setup_helper import run, conda_run, conda_python, conda_base, env_exists, section

ENV_NAME = "e2e_refinement"      # README 에서 만들어 둔 conda 환경 이름
print("REPO_ROOT :", REPO_ROOT)
print("ENV_NAME  :", ENV_NAME)

# 환경이 준비되었는지만 확인한다. 생성과 커널 등록은 README 「준비」에서 끝났다고 본다.
if conda_base() is None:
    raise SystemExit("conda 를 찾지 못했습니다. README 「준비」 1 절을 먼저 수행하십시오.")
if not env_exists(ENV_NAME):
    raise SystemExit(f"conda 환경 '{ENV_NAME}' 이 없습니다. README 「준비」 2 절을 먼저 수행하십시오.")
print("conda base :", conda_base())
conda_run(ENV_NAME, "python -V && which python && pip -V")

## 1. 시스템 점검

여기서 확인하는 것은 네 가지다.

- **glibc 버전** — acados 의 템플릿 렌더러(`t_renderer`) 기본 빌드가 GLIBC 2.34 를 요구한다.
  2.34 미만이면 설치 스크립트가 구버전 바이너리로 자동 대체한다 (4단계에서 로그로 보인다).
- **컴파일러 / cmake** — acados 를 소스 빌드하므로 필요하다. 없으면 4단계에서 `sudo apt-get` 을 시도한다.
- **GPU** — 없어도 동작하지만 스텝당 추론이 크게 느려진다.
- **디스크** — conda 환경 ~8 GB + acados ~1 GB. 데이터는 별도다.

> **파이프라인 역할**: 설치 자체는 없다. glibc 는 [C] 의 코드 생성기가, GPU 는 [B] 의
> 추론 속도가, 컴파일러는 [C] 의 acados 빌드가 각각 걸리는 지점이다.

In [ ]:
section("시스템 점검")
run("cat /etc/os-release | head -2; echo; ldd --version | head -1")
run("for c in gcc make cmake git curl; do printf '%-8s ' $c; command -v $c || echo '없음'; done")
run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo 'GPU 없음 (CPU 로 진행 가능)'")
run(f"df -h {REPO_ROOT} $HOME | awk 'NR==1 || /[0-9]%/'")

## 2. 파이썬 패키지 설치
---

---
`requirements.txt` 의 패키지를 설치한다. torch 휠이 2.2 GB 라 이 셀이 가장 오래 걸린다.

> **파이프라인 역할**: [C] 를 뺀 나머지 전부. 세부 대응은 이렇다.
>
> | 패키지 | 어디에 |
> |---|---|
> | `sqlalchemy` `geopandas` `pyogrio` `rasterio` `shapely` | [A] DB ORM · 지도 gpkg 로드 |
> | `hydra-core` `omegaconf` | [A] planner/filter/builder 설정 조합 |
> | `ray` `psutil` | [A] 시나리오 병렬 실행 |
> | `torch+cu118` `timm` | [B] 모델 forward |
> | `numba` `cv2` | [B] feature builder (`src/feature_builders/common.py`) |
> | `pytorch-lightning` `tensorboard` | [B] ckpt 로드 · devkit `timing_callback` |
> | `numpy` `scipy` | [D] forward simulation (LQR + kinematic bicycle) |
> | `matplotlib` `Pillow` `imageio` `imageio-ffmpeg` | [E] 프레임 렌더 · mp4 저장 |
> | `pandas` `pyarrow` | [F] CSV · 공식 집계 parquet |

In [ ]:
# Requirements를 이용해 설치하는 라이브러리 확인
section("requirements.txt")
run(f"head -8 {REPO_ROOT}/requirements.txt")

In [ ]:
# 10분 내외 소요. 진행 로그가 계속 흐른다.
rc = conda_run(
    ENV_NAME,
    # python -s -m pip : 환경의 python 으로 pip 를 부르고 user site 를 무시한다.
    #   bare `pip` 는 다른 환경의 것이 잡힐 수 있어 설치가 조용히 실패한다.
    f"python -s -m pip install -r {REPO_ROOT}/requirements.txt"
)

## 3. natten 설치

**natten**(*Neighborhood Attention Extension*)은 이름 그대로 **Neighborhood Attention** 을 전용
CUDA 커널로 구현한 확장이다.

일반적인 self-attention 은 모든 토큰이 모든 토큰을 본다 — 시퀀스 길이 N 에 대해 연산과 메모리가
N² 로 늘어난다. Neighborhood Attention 은 각 토큰이 **자기 주변 `kernel_size` 개**만 보도록 제한한다.

| | 전역 self-attention | Neighborhood Attention |
|---|---|---|
| 보는 범위 | 시퀀스 전체 | 자기 주변 k 개 |
| 복잡도 | N² | N·k — 길이에 선형 |
| 귀납 편향 | 없음 | **지역성** — 가까운 시점끼리 먼저 묶인다 |
| 넓은 문맥 | 처음부터 전체 | 계층을 쌓아 수용 영역을 넓힌다 (CNN 과 같은 방식) |

슬라이딩 윈도우 어텐션은 `unfold` 와 마스킹으로도 흉내 낼 수 있으나, natten 은 이를 하나의
CUDA 커널로 융합해 두어 더 빠르고 메모리를 적게 쓴다. 대신 **torch 버전과 강하게 결합**되어
`torch200cu118` 전용 빌드를 받아야 하므로 2 단계 다음에 온다.

### PLUTO 에서 쓰이는 곳

**주변 객체의 과거 궤적을 인코딩하는 곳 한 군데**다.

```
AgentEncoder.history_encoder          modules/agent_encoder.py
  └ NATSequenceEncoder                layers/embedding.py
      └ NATBlock → NATLayer
          └ natten.NeighborhoodAttention1D
```

입력은 주변 차량 한 대의 **과거 21 스텝(2 초) × 9 채널** 시계열이다. 커널 3 → 3 → 5,
채널 32 → 64 → 128 로 3 단계를 거치며 시간축을 좁혀 나가고, 최종적으로 차량 한 대가 토큰
하나가 된다. 실습 2 의 PLUTO 구조 그림에서 `E_A` 로 표시된 것이 이 토큰이다.

시계열에서 의미 있는 패턴(감속 시작, 차선 변경 개시 같은 것)은 대체로 국소적이므로, 전 구간을
전체를 한꺼번에 보는 방식보다 인접 구간부터 단계적으로 결합하는 방식이 이 입력에 적합하다.

> **파이프라인 역할**: [B] 전용이며 PLUTO 에만 쓰인다. Diffusion Planner 는 natten 을 쓰지
> 않으므로, Diffusion 어댑터만 다룰 것이라면 이 단계를 건너뛰어도 된다.


In [ ]:
section("natten")
rc = conda_run(ENV_NAME,
    "pip install natten==0.14.6+torch200cu118 -f https://whl.natten.org/old")

## 4. acados 설치 (실습 3 전용)

MPC refinement 실습(실습 3)에만 필요하다. 실습 1·2 는 이 단계 없이 진행할 수 있다.

스크립트가 하는 일:
1. 빌드 도구 확인 (이미 있으면 sudo 를 요구하지 않는다)
2. acados `v0.5.4` 를 `Trajectory_refinement/acados` 에 clone — 저장소 안이라 통째로 옮겨도 따라온다
3. cmake 로 빌드한다
4. `acados_template` 의존 패키지 확인
5. `t_renderer` 선다운로드 — 없으면 코드 생성 중 `input()` 프롬프트에서 멈춘다

> **파이프라인 역할**: [C] 전용이자 **유일한 선택 단계**. 저장소에서 acados 를 import 하는
> 파일은 `src/planners/utils/mpc_interface.py` 하나뿐이고, planner 가 이걸 try 안에서
> 지연 import 한다. 그래서 여기서 실패해도 [A][B][D][E][F] 는 그대로 돈다.

In [ ]:
section("acados")
rc = conda_run(ENV_NAME, f"bash {REPO_ROOT}/script/install_acados.sh")
if rc != 0:
    print("\n⚠ acados 설치 실패 — 5·10단계를 건너뛰어도 실습 1·2 는 진행할 수 있습니다.")

## 5. MPC 솔버 빌드

acados 로 MPC 문제를 C 코드로 생성하고 `.so` 로 컴파일한다. **빌드는 여기서 한 번만**
수행하며, 이후 시뮬레이션은 만들어진 `.so` 를 그대로 불러 쓴다.

> **파이프라인 역할**: [C] 가 매 스텝 호출하는 `.so` 를 만든다. 4단계가 acados 라이브러리를
> 깔았다면, 이 단계는 **우리 MPC 문제**(kinematic bicycle + 차선/충돌 제약)를 그 라이브러리로
> 컴파일하는 것이다. 산출물은 `Trajectory_refinement/refinementMPC/c_generated_code/`.

In [ ]:
section("MPC 솔버 빌드")
rc = conda_run(ENV_NAME, f"cd {REPO_ROOT} && bash script/build_mpc.sh")

## 6. 설치 검증

실습 경로가 실제로 import 하는 패키지를 모두 확인한다.

> **파이프라인 역할**: [A]~[F] 가 쓰는 패키지를 한 번에 확인한다. 여기서 FAIL 이 나면
> 그 패키지가 담당하는 단계에서 런이 죽는다 (위 대응표 참고).
> 두 번째 셀의 MPC 로드는 [C] 만 확인하므로, 실패해도 ML-only 로 진행할 수 있다.

In [ ]:
section("패키지 검증")
CHECK = r'''
import importlib
mods = ["torch","torchvision","pytorch_lightning","torchmetrics","timm","natten","numba",
        "imageio","imageio_ffmpeg","cv2","hydra","ray","shapely","geopandas","casadi",
        "tensorboard","numpy","pandas","rasterio","fiona"]
bad = []
for m in mods:
    try:
        x = importlib.import_module(m)
        print("  OK   %-20s %s" % (m, getattr(x, "__version__", "")))
    except Exception as e:
        bad.append(m); print("  FAIL %-20s %s" % (m, e))
import torch
print("\n  torch.cuda.is_available() =", torch.cuda.is_available())
print("  결과:", "전부 정상" if not bad else f"누락 {bad}")
'''
conda_python(ENV_NAME, CHECK)

In [ ]:
# acados / MPC 솔버는 별도로 확인한다 (없어도 ML-only 로 진행 가능).
section("acados + MPC 솔버")
conda_python(ENV_NAME,
    "from src.planners.utils.mpc_interface import RefinementMpcInterface\n"
    "RefinementMpcInterface()\n"
    "print('MPC 로드 OK')\n",
    setup_env=True, quiet_ok=True)

## 7. 데이터 배치 확인

데이터는 저장소에 포함되지 않는다. 아래 구조로 직접 넣어야 한다.

```
data/db/<이름>/*.db                 시나리오 DB
data/maps/<지도명>/.../map.gpkg     지도 (gpkg 만 있으면 된다. .npy 캐시는 devkit 이 만든다)
data/model/pluto_planner.ckpt       PLUTO 체크포인트
```

**DB 폴더를 새로 추가하면 devkit yaml 에 경로를 한 줄 더해야 한다.**
devkit 의 DB 탐색은 `Path.iterdir()` 기반이라 깊이 1까지만 본다 — 하위 폴더를 재귀하지
않으므로, `data/db` 만 지정하면 예외 없이 조용히 0개가 된다.

> **파이프라인 역할**: [A] 의 입력이다. 패키지가 다 깔려 있어도 여기가 비면 시나리오 0개로
> 런이 조용히 끝난다 — 에러가 아니라 "할 일이 없음" 으로 처리되므로 8단계에서 꼭 확인한다.

In [ ]:
section("데이터 배치")
run(f"cd {REPO_ROOT} && echo '--- data/db ---' && ls -d data/db/*/ 2>/dev/null || echo '  (없음)'")
run(f"cd {REPO_ROOT} && echo '--- DB 파일 수 ---' && find data/db -name '*.db' 2>/dev/null | wc -l")
run(f"cd {REPO_ROOT} && echo '--- 지도 ---' && find data/maps -name 'map.gpkg' 2>/dev/null || echo '  (없음)'")
run(f"cd {REPO_ROOT} && echo '--- 모델 ---' && ls -lh data/model/ 2>/dev/null || echo '  (없음)'")
run(f"cd {REPO_ROOT} && echo '--- devkit 이 보는 DB 경로 ---' && "
    "grep -A4 '^data_root' nuplan/planning/script/config/common/scenario_builder/nuplan.yaml")

## 8. 시나리오 빌드 확인

환경변수를 잡고 Hydra 설정이 조합되는지, 필터의 토큰이 실제 DB 에서 찾아지는지 본다.
여기까지 통과하면 시뮬레이션을 돌릴 수 있는 상태다.

> **파이프라인 역할**: [A] 를 끝까지 한 번 실행해 보는 것이다. **요청 토큰 수와 빌드된
> 시나리오 수가 같아야 한다.** 빌드 수가 적으면 그 토큰의 DB 가 `data/db/` 에 없거나
> devkit yaml 의 `data_root` 에 폴더가 빠진 것이다.

In [ ]:
section("환경변수")
conda_run(ENV_NAME,
    f"cd {REPO_ROOT} && source script/nuplan_env.sh && "
    "env | grep -E '^(NUPLAN_|PRACTICE_MODEL|ACADOS_SOURCE)' | sort")

In [ ]:
section("시나리오 빌드")
BUILD = r'''
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf
from nuplan.planning.script.builders.scenario_building_builder import build_scenario_builder
from nuplan.planning.script.builders.scenario_filter_builder import build_scenario_filter
from nuplan.planning.utils.multithreading.worker_sequential import Sequential

for f in ["practice_scenarios"]:
    GlobalHydra.instance().clear()
    with initialize(config_path="./config"):
        cfg = compose(config_name="default_simulation", overrides=[
            "+simulation=closed_loop_nonreactive_agents", "planner=refinement_planner",
            "planner/model_adapter=pluto", "scenario_builder=nuplan",
            f"scenario_filter={f}", "worker=sequential"])
    # build_scenario_filter 는 전체 cfg 가 아니라 scenario_filter 노드를 받는다.
    n_req = len(OmegaConf.select(cfg, "scenario_filter.scenario_tokens") or [])
    sc = build_scenario_builder(cfg).get_scenarios(
        build_scenario_filter(cfg.scenario_filter), Sequential())
    print("  %-18s 요청 %2d 토큰 -> 빌드 %2d 시나리오" % (f, n_req, len(sc)))
'''
conda_python(ENV_NAME, BUILD, setup_env=True)

## 9. Open-loop 스모크 런 — nuPlan 기본 planner

여기까지는 부품이 갖추어졌는지만 확인했다. 이제 **시뮬레이션 한 번을 끝까지** 돌린다.
모델도 acados 도 쓰지 않고 devkit 내장 planner 로만 돌리므로 1 분이면 끝나며, 실습 1 이
사용하는 경로와 같다.

두 planner 를 Open-loop(`open_loop_boxes`)로 돌린다. Open-loop 는 계획 궤적이 로그의 실제 궤적과
얼마나 닮았는지만 재므로, 결과를 미리 알 수 있다.

| planner | 내는 궤적 | 기대 점수 |
|---|---|---|
| `log_future_planner` | 로그의 실제 미래를 그대로 출력 (정답) | **1.000** |
| `simple_planner` | 현재 방향으로 직진만 | **0.000** |

> **파이프라인 역할**: 시나리오 빌드 → 시뮬레이션 루프 → 공식 지표 → 집계까지 한 번에
> 통과시킨다. 정답을 넣었는데 1.000 이 아니면 채점·집계 어딘가가 어긋난 것이므로,
> 이 단계가 통과해야 이후 실습의 점수를 신뢰할 수 있다.


In [ ]:
section("Open-loop 스모크 런 (planner 2종, 각 ~40초)")
for p in ["log_future_planner", "simple_planner"]:
    conda_run(ENV_NAME,
        f"cd {REPO_ROOT} && source script/nuplan_env.sh && "
        f"python run_simulation.py +simulation=open_loop_boxes planner={p} "
        "scenario_builder=nuplan scenario_filter=practice_scenarios "
        "scenario_filter.limit_total_scenarios=1 worker=sequential "
        f"experiment_uid=practice0/open_loop/{p}")


In [ ]:
section("Open-loop 스모크 런 결과")
CHECK = r"""
import glob
import pandas as pd

BASE = "data/exp/simulation/open_loop_boxes/practice0/open_loop"
COLS = ["planner_expert_average_l2_error_within_bound",
        "planner_expert_final_l2_error_within_bound",
        "planner_miss_rate_within_bound"]

rows = []
for p in ["log_future_planner", "simple_planner"]:
    files = sorted(glob.glob(f"{BASE}/{p}/aggregator_metric/*.parquet"))
    if not files:
        print(f"  {p}: 집계 parquet 이 없습니다 — 위 실행 로그를 확인하십시오.")
        continue
    r = pd.read_parquet(files[-1]).query("scenario == 'final_score'").iloc[0]
    rows.append({"planner": p, "final_score": round(float(r["score"]), 3),
                 **{c.replace("planner_expert_", "").replace("_within_bound", ""):
                    round(float(r[c]), 3) for c in COLS}})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

ok = dict(zip(df["planner"], df["final_score"]))
if ok.get("log_future_planner") == 1.0 and ok.get("simple_planner") == 0.0:
    print("\n정답 planner 1.000 / 직진 planner 0.000 — 채점과 집계가 정상입니다.")
else:
    print("\n기대값(1.000 / 0.000)과 다릅니다. 시나리오나 지표 설정을 확인하십시오.")
"""
conda_python(ENV_NAME, CHECK, setup_env=True)


**확인 사항** — `simple_planner` 의 0.000 은 실패가 아니다. Open-loop 점수는
`planner_miss_rate_within_bound` 를 곱셈 항으로 쓰는데, 직진 궤적은 허용 오차를 크게 벗어나
이 항이 0 이 되기 때문이다. 두 값이 각각 1.000 과 0.000 으로 구분되는 것 자체가 채점 경로가
살아 있다는 증거다.

여기까지 통과했다면 **실습 1 을 진행할 수 있다.** 실습 1 은 지금 돌린 것과 같은 명령을
planner 를 바꿔 가며 실행하고, Open-loop 와 Closed-loop 의 차이를 다룬다.
아래 10 단계는 모델과 MPC 를 쓰는 실습 2 이후를 위한 것이므로 지금 건너뛰어도 된다.


## 10. 모델 스모크 런 (실습 2 이후)

PLUTO 체크포인트와 acados 까지 포함해 전체 경로(추론 → MPC → 채점 → 렌더)가 도는지 본다.
`worker=sequential` 이라 느리다 — 스텝당 ~5 초. 몇 스텝 로그가 보이면 성공이므로
중간에 **커널 인터럽트(정지 버튼)** 로 끊어도 된다.

실습 1 은 이 단계 없이도 진행할 수 있다. 모델이 필요한 것은 실습 2 부터다.

> **파이프라인 역할**: 9 단계가 devkit 만 확인했다면, 여기는 모델과 MPC 까지 확인한다.
> 아래 셀의 CSV 에 `ml_final` 과 `rf_final` 이 **둘 다** 차 있으면 MPC 까지 살아 있다는 뜻이고,
> `rf_final` 이 비어 있으면 ML-only 로 돌고 있는 것이다(실습 2 는 그 상태로도 충분하다).


In [ ]:
section("스모크 런 (Ctrl+C / 정지 버튼으로 중단 가능)")
try:
    conda_run(ENV_NAME,
        f"cd {REPO_ROOT} && EXP_STAMP=practice0 "
        "bash script/run_refinement_planner.sh pluto ml practice_scenarios 0")
except KeyboardInterrupt:
    print("\n중단했습니다 — 아래 셀에서 남은 결과를 확인하세요.")

In [ ]:
section("스텝별 채점 결과")
STEPS = r"""
import glob
import pandas as pd

files = sorted(glob.glob("data/exp/simulation/**/practice0_*/**/*.csv", recursive=True))
if not files:
    print("CSV 가 아직 없습니다. 스모크 런을 몇 스텝 더 돌려 주세요.")
else:
    df = pd.read_csv(files[-1])
    print(f"{files[-1]}\n스텝 {len(df)}개")
    cols = [c for c in ["iteration", "driving_policy", "executed", "refine_reason",
                        "ml_final", "rf_final", "delta"] if c in df.columns]
    print(df[cols].head(10).to_string(index=False))
"""
conda_python(ENV_NAME, STEPS, setup_env=True)

## 완료

환경 구성이 끝났다. **[practice1_nuplan_framework.ipynb](practice1_nuplan_framework.ipynb) 부터
순서대로** 진행한다.

| 실습 | 다루는 것 | 필요한 것 |
|---|---|---|
| 1 | nuPlan 프레임워크 — 시나리오·컨트롤러·시뮬레이션 루프, Open-loop 와 Closed-loop | 9 단계까지 |
| 2 | ML planner(PLUTO) Open-loop·Closed-loop 평가 | + 체크포인트, GPU |
| 3 | MPC refinement *(준비 중)* | + acados |

### 잘 안 될 때

| 증상 | 원인 / 조치 |
|---|---|
| `No matching distribution found for aioboto3` | `requirements.txt` 가 `--index-url` 로 되어 있다. `--extra-index-url` 이어야 한다 |
| `TypeError: unhashable type: 'list'` (aiohttp) | 환경이 Python 3.9.0 이다. README 「준비」 2 절을 `python=3.9` 로 다시 수행한다 |
| `No module named 'tensorboard'` | 2단계가 끝까지 돌지 않았다. 다시 실행한다 |
| `Rendering file main.in.c failed` | `t_renderer` 가 glibc 와 안 맞는다. 4단계를 다시 돌리면 구버전으로 자동 대체된다 |
| `No module named 'deprecated'` | `acados_template` 의존성 누락. 4단계를 다시 실행한다 |
| 시나리오 0개 빌드 | DB 폴더가 devkit `nuplan.yaml` 의 `data_root` 에 없다 (7단계 참고) |
| Open-loop 스모크 런 점수가 1.000/0.000 이 아님 | 시나리오 필터나 지표 설정이 바뀌었다 (9단계 참고) |
| 그래프의 한글이 빈 네모로 나옴 | 한글 폰트가 없다. `sudo apt install fonts-nanum` 후 커널 재시작 |